# Train merge-ME-basic DQN_ME on Colab

Clones a selected Git ref of `RQL-Comparison`, creates an isolated virtualenv, and runs the same training command used for the highway prior:

```bash
python train_rl.py --env-name merge_basic --algo dqn_ME --rollout-save-n-episodes 1000
```

Uses `merge-ME-basic-v0` with the merge basic-reward config and highway-matched DQN hyperparams (`lr=1e-4`, `5e5` timesteps by default).

**Checkpoints write directly to Google Drive** under `models/{RUN_NAME}/` (every 10k timesteps), so a Colab disconnect does not wipe progress. Keep `RUN_NAME` fixed and set `RESUME=True` to continue from the latest Drive checkpoint.

**Before running:** push the merge env / `merge_basic` named config / `train_rl.py --log-dir` support to the Git ref you select below. Prefer a GPU runtime.

In [1]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
# Training parameters (edit this cell before running).
REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

ENV_NAME = "merge_basic"  #@param {type:"string"}
ALGO = "dqn_ME"  #@param {type:"string"}
TOTAL_TIMESTEPS = 500000  #@param {type:"integer"}
ROLLOUT_SAVE_N_EPISODES = 1000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}

# Keep RUN_NAME fixed across reconnects so checkpoints land in the same Drive folder.
RUN_NAME = "merge_basic_final_seed1"  #@param {type:"string"}
DRIVE_MODELS_ROOT = "/content/drive/MyDrive/aaai2027/RQL-Comparison/models"  #@param {type:"string"}
RESUME = True  #@param {type:"boolean"}

In [12]:
# Clone the requested ref and create a Colab-compatible training environment.
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
VENV_DIR = Path("/content/venvs/rql-comparison-train")

# Headless Colab rendering for pygame / highway-env.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)


def create_venv(venv_dir: Path) -> Path:
    """Create a Colab-friendly venv that reuses system site packages (incl. CUDA torch)."""
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)

# Packages needed by train_rl.py / DQN_ME on current Colab Python.
# Do not pin numpy==1.23.5: it has no Python 3.12 wheels.
# Use gym==0.26.2 (not 0.23.1): Drive checkpoints pickle spaces with
# RandomNumberGenerator; gym 0.23 cannot unpickle them on resume.
# Local stable_baselines3 and highway_env are imported from PROJECT_DIR via cwd/PYTHONPATH.
TRAIN_PACKAGES = [
    "numpy<2", "gym==0.26.2", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml", "sacred",
    # Required by imitation.policies.serialize (imported from train_rl.py).
    "huggingface_sb3>=2.2.1,<3",
]
result = subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", *TRAIN_PACKAGES],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, result.args)

entry_point = PROJECT_DIR / "train_rl.py"
merge_env = PROJECT_DIR / "highway_env" / "envs" / "merge_env.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
if "MergeEnvMEBasic" not in merge_env.read_text():
    raise RuntimeError(
        f"{merge_env} does not define MergeEnvMEBasic. "
        f"Checkout a ref that includes the merge-ME-basic training changes."
    )
if "--log-dir" not in entry_point.read_text():
    raise RuntimeError(
        f"{entry_point} is missing --log-dir support. "
        f"Push the updated train_rl.py before running this notebook."
    )

print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")
print(f"Ref:     {REPO_REF}")

stdlib venv failed; falling back to virtualenv
Error: Command '['/content/venvs/rql-comparison-train/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.

Project: /content/RQL-Comparison
Python:  /content/venvs/rql-comparison-train/bin/python
Ref:     main


In [ ]:
# Run train_rl.py with log_dir on Drive so checkpoints survive disconnects.
import json
import os
from pathlib import Path

RUN_DIR = Path(DRIVE_MODELS_ROOT).expanduser() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "env_name": ENV_NAME,
    "algo": ALGO,
    "total_timesteps": TOTAL_TIMESTEPS,
    "rollout_save_n_episodes": ROLLOUT_SAVE_N_EPISODES,
    "seed": SEED,
    "run_dir": str(RUN_DIR),
    "resume": RESUME,
    "command": (
        f"python train_rl.py --env-name {ENV_NAME} --algo {ALGO} "
        f"--rollout-save-n-episodes {ROLLOUT_SAVE_N_EPISODES} "
        f"--total-timesteps {TOTAL_TIMESTEPS} --seed {SEED} "
        f"--log-dir {RUN_DIR}"
    ),
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON),
    str(PROJECT_DIR / "train_rl.py"),
    "--env-name", ENV_NAME,
    "--algo", ALGO,
    "--rollout-save-n-episodes", str(ROLLOUT_SAVE_N_EPISODES),
    "--total-timesteps", str(TOTAL_TIMESTEPS),
    "--seed", str(SEED),
    "--log-dir", str(RUN_DIR),
]

policy_dir = RUN_DIR / "policies"
has_checkpoint = policy_dir.is_dir() and any(policy_dir.rglob("model.zip"))
if RESUME and has_checkpoint:
    command += ["--resume-from-logdir", str(RUN_DIR)]
    print(f"Resuming from latest checkpoint under {policy_dir}")
elif RESUME and not has_checkpoint:
    print(f"No prior checkpoint in {policy_dir}; starting a new run.")

print(f"Drive run dir: {RUN_DIR}")
print("$", " ".join(command))

env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": str(PROJECT_DIR),
    "SDL_VIDEODRIVER": "dummy",
    "OFFSCREEN_RENDERING": "1",
}

log_mode = "a" if RESUME and (RUN_DIR / "training.log").exists() else "w"
with (RUN_DIR / "training.log").open(log_mode, buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=str(PROJECT_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

final_policy = RUN_DIR / "policies" / "final" / "model.zip"
print(
    f"Final policy: {final_policy}  "
    f"({'ok' if final_policy.is_file() else 'MISSING / still training checkpoints only'})"
)
print(f"Results root: {RUN_DIR}")

No prior checkpoint in /content/drive/MyDrive/aaai2027/RQL-Comparison/models/merge_basic_final_seed1/policies; starting a new run.
Drive run dir: /content/drive/MyDrive/aaai2027/RQL-Comparison/models/merge_basic_final_seed1
$ /content/venvs/rql-comparison-train/bin/python /content/RQL-Comparison/train_rl.py --env-name merge_basic --algo dqn_ME --rollout-save-n-episodes 1000 --total-timesteps 500000 --seed 1 --log-dir /content/drive/MyDrive/aaai2027/RQL-Comparison/models/merge_basic_final_seed1
/content/RQL-Comparison/train_rl.py:13: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not hasattr(np, "bool8"):
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import

In [ ]:
# Verify the artifacts persisted to Drive (including intermediate policy checkpoints).
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")